In [1]:
# 공통 설정
# .env 파일에서 API Key와 기본 모델명을 읽어온다.
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPEAI_API_KEY를 환경 변수로 설정하세요.")

client = OpenAI(api_key=api_key)
DEFAULT_MODEL = os.getenv("OPENAI_DEFAULT_MODEL", "gpt-4.1-mini")

print("OpenAI client 준비 완료")
print("기본 모델 : ", DEFAULT_MODEL)

OpenAI client 준비 완료
기본 모델 :  gpt-4.1-mini


# Streaming
일반 응답 방식은 모델이 전체 답변을 생성한 뒤 한 번에 결과를 돌려준다.
코드가 단순하고 후처리가 쉽지만, 답변이 긴 경우 사용자는 기다리는 시간이 길게 느껴질 수 있다.

In [2]:
import time

start = time.perf_counter()

response = client.responses.create(
    model=DEFAULT_MODEL,
    instructions='너는 초급 개발자에게 쉽게 설명하는 AI 강사이다.',
    input='Streaming 응답이 필요한 이유를 설명해줘.'
)

elapsed = time.perf_counter() - start

print(response.output_text)
print(f"elapsed : {elapsed:.2f}s")

좋아요! 초급 개발자에게 쉽게 설명해줄게요.

**Streaming 응답이 필요한 이유**는 크게 3가지예요:

1. **빠른 사용자 경험**  
   데이터를 한 번에 다 받을 때까지 기다리지 않고, 조금씩 받아서 바로 보여줄 수 있어요. 예를 들어 채팅 앱에서 긴 메시지를 쓸 때, 문장이 한 문장씩 바로바로 화면에 뜨면 더 자연스럽고 빠르게 느껴지죠.

2. **효율적인 네트워크 사용**  
   큰 데이터를 한꺼번에 보내면 서버와 클라이언트가 부담을 느껴요. Streaming은 데이터를 쪼개서 조금씩 보내니까, 네트워크에 과부하가 적고 더 안정적이에요.

3. **실시간 처리 가능**  
   서버에서 처리 결과를 조금씩 생성할 때마다 바로바로 클라이언트로 보내면, 실시간으로 동작하는 느낌을 줄 수 있어요. 예를 들어, AI 챗봇 답변을 한 글자씩 보여주는 것도 Streaming 덕분이에요.

요약하면, Streaming 응답은 **"빠르게 보여주고, 네트워크를 효율적으로 쓰고, 실시간 느낌을 주기 위해"** 필요합니다!

더 궁금한 점 있으면 물어봐 주세요!
elapsed : 9.41s


In [10]:
# Streaming 응답 방식 적용
stream = client.responses.create(
    model=DEFAULT_MODEL,
    instructions="너는 초급 개발자에게 쉽게 설명하는 AI 강사이다.",
    input="Streaming 응답을 식당 주문 처리에 비유해서 설명해줘.",
    stream=True
)

for event in stream:
    # delta 이벤트는 새로 생성 된 텍스트 조각을 의미한다.
    if event.type == 'response.output_text.delta':
        print(event.delta, end="", flush=True)

물론이야! Streaming 응답을 식당 주문 처리에 비유해서 쉽게 설명해볼게.

---

### 식당 주문 처리 비유

1. **일반적인 응답 (한꺼번에 주문 다 받기)**  
손님이 식당에 와서 주문을 할 때, "김치찌개 1인분, 불고기 1인분, 된장찌개 1인분 주세요!" 하고 한꺼번에 주문을 다 하면  
주방에서는 주문이 다 들어온 걸 확인한 뒤에, 모든 음식이 다 완성된 후에 한꺼번에 손님에게 음식을 가져다 줘.  
- 즉, **요청 → 처리 완료 후 한꺼번에 응답**

2. **Streaming 응답 (요리 하나씩 순서대로 배달하기)**  
반면에, Streaming 응답은 손님이 같은 주문을 했지만, 주방에서 김치찌개부터 빠르게 만들어서 바로 바로 가져다 주는 거야.  
김치찌개가 나오면 바로 받고, 그 다음 불고기가 완성되면 또 받고, 계속 음식이 준비되는 대로 바로바로 받아서 먹을 수 있지.  
- 즉, **요청 → 처리 중에도 조금씩 내려받으면서 응답**

---

### 그래서 Streaming 응답의 장점은?

- 손님(사용자)이 음식(데이터)을 기다리지 않고 조금씩 받아서 바로바로 먹을 수 있어서 더 빨리 만족할 수 있어!  
- 만약 불고기 준비가 오래 걸려도, 김치찌개부터 먼저 먹을 수 있으니까 대기 시간이 지루하지 않아.

---

이처럼, **Streaming 응답은 서버가 데이터를 준비하는대로 조금씩 보내주는 방식**이고,  
식당에서 요리가 완성되는 순서대로 빠르게 손님에게 음식을 내오는 느낌이라고 이해하면 쉬워!  

필요하면 더 자세히도 알려줄게~

# 토큰 사용량 확인
API 응답 객체에는 입력 토큰, 출력 토큰, 전체 토큰 정보를 담은 `usage` 가 포함 된다.

In [12]:
response = client.responses.create(
    model=DEFAULT_MODEL,
    instructions='너는 간결하게 답하는 AI 강사이다.',
    input='토큰이 무엇인지 설명해줘.'
)

print(response.output_text)
print()
print(response.usage)

토큰은 자연어 처리에서 문장이나 텍스트를 나눈 최소 단위입니다. 예를 들어, 단어, 구두점, 심볼 등이 토큰이 될 수 있습니다.

ResponseUsage(input_tokens=32, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=43, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=75)


## 비용 추정 함수

In [13]:
PRICE_PER_1M_INPUT = 0.40
PRICE_PER_1M_OUTPUT = 1.60

def estimate_cost(usage):
    """
    Responses API usage 객체를 받아 예상 비용을 계산하는 함수
    """
    if usage is None:
        return None
    
    input_tokens = getattr(usage, 'input_tokens', 0) or 0
    output_tokens = getattr(usage, "output_tokens", 0) or 0

    return (input_tokens / 1_000_000) * PRICE_PER_1M_INPUT + (output_tokens / 1_000_000) * PRICE_PER_1M_OUTPUT

cost = estimate_cost(response.usage)
print("estimate cost($) : ", cost)

estimate cost($) :  8.16e-05


## API 호출 로그 남기기
실험을 많이 할 수록 어떤 프롬프트가 좋았는지 기억하기 어렵다.
따라서 프롬프트, 모델명, 파라미터, 응답 시간, 토큰 사용량, 비용, 출력 결과등을 표로 남긴다.

In [ ]:
import pandas as pd 

logs = []

def logged_response(prompt, instructions, model=DEFAULT_MODEL, temperature=0.3):
    start = time.perf_counter()
    response = client.responses.create(
        model=model,
        instructions=instructions,
        input=prompt,
        temperature=temperature
    )
    elapsed = time.perf_counter() - start
    usage = response.usage

    row = {
        "model" : model,
        "temperature" : temperature,
        "prompt" : prompt,
        "output" : response.output_text,
        "elapsed_sec" : round(elapsed, 3),
        "input_tokens" : getattr(usage, "input_tokens", None) if usage else None,
        "output_tokens" : getattr(usage, "output_tokens", None) if usage else None,
        "total_tokens" : getattr(usage, "total_tokens", None) if usage else None,
        "estimate_cost_usd" : estimate_cost(usage)
    }
    logs.append(row)
    return response.output_text

In [15]:
prompts = [
    'RAG를 한 문장으로 설명해줘.',
    'RAG를 초급 개발자에게 5문장으로 설명해줘.',
    'RAG를 백엔드 개발자의 관점에서 설명해줘.',
]

for p in prompts:
    print(logged_response(p, instructions="너는 AI강사다.", temperature=0.2))
    print("-" * 60)

log_df = pd.DataFrame(logs)
log_df

RAG는 외부 지식베이스에서 정보를 검색한 후 이를 바탕으로 생성 모델이 답변을 생성하는 하이브리드 AI 기법입니다.
------------------------------------------------------------
RAG는 "Retrieval-Augmented Generation"의 약자예요. 먼저, 컴퓨터가 필요한 정보를 외부 데이터베이스나 문서에서 찾아와요. 그런 다음, 찾아온 정보를 바탕으로 자연스러운 문장을 생성해 답변을 만들죠. 이렇게 하면 모델이 모르는 내용도 정확하게 답할 수 있어요. 즉, 검색과 생성 기능을 결합한 기술이라고 이해하면 돼요.
------------------------------------------------------------
물론입니다! 백엔드 개발자의 관점에서 RAG(Retrieval-Augmented Generation)를 설명해드릴게요.

---

### RAG란?

RAG는 **Retrieval-Augmented Generation**의 약자로, 기존의 생성형 AI(예: GPT) 모델에 **정보 검색(Retrieval)** 기능을 결합한 기술입니다. 즉, 모델이 단순히 학습된 지식만으로 답변을 생성하는 것이 아니라, 외부 데이터베이스나 문서 저장소에서 관련 정보를 먼저 검색(Retrieval)한 뒤, 그 정보를 바탕으로 답변을 생성(Generation)합니다.

---

### 백엔드 개발자의 관점에서 RAG의 구성 요소

1. **인덱싱 및 검색 시스템 (Retriever)**
   - 대량의 문서, 데이터, 지식 베이스를 효율적으로 검색할 수 있도록 인덱싱합니다.
   - 보통 Elasticsearch, FAISS, Pinecone 같은 벡터 검색 엔진을 사용합니다.
   - 쿼리(사용자 질문)를 받아 관련 문서나 텍스트 조각을 빠르게 찾아냅니다.

2. **생성 모델 (Generator)**
   - 검색된 문서들을 입력으로 받아, 자연어 생성 모델(예: GPT, T5)이 최종 답변을 생성합니다.


,model,temperature,prompt,output,elapsed_sec,input_tokens,output_tokens,total_tokens,estimate_cost_usd:
0,gpt-4.1-mini,0.2,RAG를 한 문장으로 설명해줘.,RAG는 외부 지식베이스에서 정보를 검색한 후 이를 바탕으로 생성 모델이 답변을 생...,2.221,29,37,66,0.000071
1,gpt-4.1-mini,0.2,RAG를 초급 개발자에게 5문장으로 설명해줘.,"RAG는 ""Retrieval-Augmented Generation""의 약자예요. 먼...",4.745,35,93,128,0.000163
2,gpt-4.1-mini,0.2,RAG를 백엔드 개발자의 관점에서 설명해줘.,물론입니다! 백엔드 개발자의 관점에서 RAG(Retrieval-Augmented G...,9.658,33,763,796,0.001234
